In [1]:
import pandas as pd
import time
pd.set_option('display.max_rows', 100)

In [2]:
start=time.perf_counter()

In [3]:
from DatabaseCreator import DatabaseCreator
import CJDH_local_settings

#Run Database Creator
if __name__ == "__main__":
    db_creator = DatabaseCreator(db_settings=CJDH_local_settings.local_settings['FPL_Points_Predictor'])
    fpl_engine = db_creator.get_engine_for("fpl_data_analysis")

In [4]:
stats_current_gameweek = 20
picks_current_gameweek = 20
# last_completed_gameweek = 13
# current_gameweek = 14
# next_gameweek = 15
current_season = 20252026

In [5]:
def create_running_averages(db_creator, 
                           metrics_to_average,
                           window_sizes=[5, 10],
                           include_alltime=True,
                           include_per_90=True,
                           include_raw=True,
                           include_elo_adjusted=True,
                           include_squared_per_90=True,
                           interaction_pairs=None,
                           additional_fields=None):
    """
    Create running averages for specified metrics with optional normalization.
    Stops calculations at the most recent gameweek with total_points data.
    
    Parameters:
    -----------
    include_elo_adjusted : bool, optional
        If True, adjust metrics by opponent ELO difficulty
        Uses ~30% adjustment per 100 ELO difference
    include_squared_per_90 : bool, optional
        If True, include squared per-90 features for capturing non-linear effects
    interaction_pairs : list of tuples, optional
        List of (metric1, metric2) pairs to create interaction features
        Example: [('expected_goals', 'expected_assists'), ('bps', 'minutes')]
        Creates features like metric1_x_metric2_per90_running_avg_prev_{window}
    """
    
    # Base fields that are always included
    base_fields = [
        'player_name_id', 'element', 'season', 'value', 'event', 'minutes', 'total_points',
        'team_elo', 'opp_team_elo', 'position', 'goals_scored', 'bonus',
        'bps', 'clean_sheets', 'goals_conceded', 'was_home',
        'expected_goals', 'expected_assists', 'expected_goal_involvements',
        'expected_goals_conceded','team_name','opp_team_name', 
        'cbi','defensive_contribution','recoveries','tackles',
        'saves'
    ]
    
    if additional_fields:
        base_fields.extend(additional_fields)
    
    select_parts = base_fields.copy()
    
    # Season totals and derived per-90 stats
    select_parts.append('SUM(minutes) OVER (PARTITION BY player_name_id, season) AS player_season_minutes_total')
    select_parts.append("""(SUM(total_points) OVER (PARTITION BY player_name_id, season) / 
                            NULLIF(SUM(minutes) OVER (PARTITION BY player_name_id, season), 0) * 90.0)
                                AS player_season_points_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (total_points / minutes * 90.0) ELSE 0 END AS points_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (expected_goals / minutes * 90.0) ELSE 0 END AS expected_goals_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (expected_assists / minutes * 90.0) ELSE 0 END AS expected_assists_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (expected_goals_conceded / minutes * 90.0) ELSE 0 END AS expected_goals_conceded_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (saves / minutes * 90.0) ELSE 0 END AS saves_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (bps / minutes * 90.0) ELSE 0 END AS bps_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (bonus / minutes * 90.0) ELSE 0 END AS bonus_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (cbi / minutes * 90.0) ELSE 0 END AS cbi_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (defensive_contribution / minutes * 90.0) ELSE 0 END AS defensive_contribution_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (recoveries / minutes * 90.0) ELSE 0 END AS recoveries_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (tackles / minutes * 90.0) ELSE 0 END AS tackles_per90""")

    select_parts.append('team_elo - opp_team_elo AS elo_diff')

    # Interaction features (product of two metrics)
    if interaction_pairs:
        for metric1, metric2 in interaction_pairs:
            # Shorten metric names for interactions
            m1 = metric1.replace('expected_', 'x').replace('_contribution', '_cont')
            m2 = metric2.replace('expected_', 'x').replace('_contribution', '_cont')
            interaction_name = f"{m1}_x_{m2}"
            
            # All-time interaction averages
            if include_alltime:
                if include_per_90:
                    metric1_per90 = f"CASE WHEN minutes > 0 THEN ({metric1} / minutes * 90.0) ELSE 0 END"
                    metric2_per90 = f"CASE WHEN minutes > 0 THEN ({metric2} / minutes * 90.0) ELSE 0 END"
                    interaction_expr = f"({metric1_per90}) * ({metric2_per90})"
                    avg_name = f"{interaction_name}_p90_at"
                    select_parts.append(f"""
                        CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                            AVG(CASE WHEN total_points IS NOT NULL 
                                THEN {interaction_expr} ELSE NULL END) OVER (
                                PARTITION BY player_name_id
                                ORDER BY season ASC, event ASC
                                ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
                            )
                        END AS {avg_name}""")
                
                # Raw interaction (not per-90)
                if include_raw:
                    raw_interaction_expr = f"{metric1} * {metric2}"
                    avg_name_raw = f"{interaction_name}_at"
                    select_parts.append(f"""
                        CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                            AVG(CASE WHEN total_points IS NOT NULL 
                                THEN {raw_interaction_expr} ELSE NULL END) OVER (
                                PARTITION BY player_name_id
                                ORDER BY season ASC, event ASC
                                ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
                            )
                        END AS {avg_name_raw}""")
            
            # Window-based interaction averages
            for window in window_sizes:
                if include_per_90:
                    metric1_per90 = f"CASE WHEN minutes > 0 THEN ({metric1} / minutes * 90.0) ELSE 0 END"
                    metric2_per90 = f"CASE WHEN minutes > 0 THEN ({metric2} / minutes * 90.0) ELSE 0 END"
                    interaction_expr = f"({metric1_per90}) * ({metric2_per90})"
                    avg_name = f"{interaction_name}_p90_{window}"
                    select_parts.append(f"""
                        CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                            AVG(CASE WHEN total_points IS NOT NULL
                                THEN {interaction_expr} ELSE NULL END) OVER (
                                PARTITION BY player_name_id
                                ORDER BY season ASC, event ASC
                                ROWS BETWEEN {window} PRECEDING AND 1 PRECEDING
                            )
                        END AS {avg_name}""")
                
                if include_raw:
                    raw_interaction_expr = f"{metric1} * {metric2}"
                    avg_name_raw = f"{interaction_name}_{window}"
                    select_parts.append(f"""
                        CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                            AVG(CASE WHEN total_points IS NOT NULL
                                THEN {raw_interaction_expr} ELSE NULL END) OVER (
                                PARTITION BY player_name_id
                                ORDER BY season ASC, event ASC
                                ROWS BETWEEN {window} PRECEDING AND 1 PRECEDING
                            )
                        END AS {avg_name_raw}""")
                
                # ELO-adjusted interaction
                if include_elo_adjusted and include_per_90:
                    elo_difficulty_multiplier = f"POWER(1.3, (opp_team_elo - team_elo) / 100.0)"
                    metric1_elo_per90 = f"CASE WHEN minutes > 0 THEN (({metric1} / minutes * 90.0) / {elo_difficulty_multiplier}) ELSE 0 END"
                    metric2_elo_per90 = f"CASE WHEN minutes > 0 THEN (({metric2} / minutes * 90.0) / {elo_difficulty_multiplier}) ELSE 0 END"
                    elo_interaction_expr = f"({metric1_elo_per90}) * ({metric2_elo_per90})"
                    avg_name_elo = f"{interaction_name}_elo_p90_{window}"
                    select_parts.append(f"""
                        CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                            AVG(CASE WHEN total_points IS NOT NULL
                                THEN {elo_interaction_expr} ELSE NULL END) OVER (
                                PARTITION BY player_name_id
                                ORDER BY season ASC, event ASC
                                ROWS BETWEEN {window} PRECEDING AND 1 PRECEDING
                            )
                        END AS {avg_name_elo}""")

    # Running average calculations
    for metric in metrics_to_average:
        # Shorten metric name
        short_metric = metric.replace('expected_', 'x').replace('_contribution', '_cont')
        
        # All-time averages
        if include_alltime:
            avg_name = f"{short_metric}_at"
            select_parts.append(f"""
                CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                    AVG(CASE WHEN total_points IS NOT NULL 
                        THEN {metric} ELSE NULL END) OVER (
                        PARTITION BY player_name_id
                        ORDER BY season ASC, event ASC
                        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
                    )
                END AS {avg_name}""")

            if include_per_90:
                metric_expr = f"CASE WHEN minutes > 0 THEN ({metric} / minutes * 90.0) ELSE 0 END"
                avg_name_per90 = f"{short_metric}_p90_at"
                select_parts.append(f"""
                    CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                        AVG(CASE WHEN total_points IS NOT NULL 
                            THEN {metric_expr} ELSE NULL END) OVER (
                            PARTITION BY player_name_id
                            ORDER BY season ASC, event ASC
                            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
                        )
                    END AS {avg_name_per90}""")
                
                # Squared per-90 all-time average
                if include_squared_per_90:
                    squared_expr = f"CASE WHEN minutes > 0 THEN POWER(({metric} / minutes * 90.0), 2) ELSE NULL END"
                    avg_name_squared = f"{short_metric}_p90sq_at"
                    select_parts.append(f"""
                        CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                            AVG(CASE WHEN total_points IS NOT NULL 
                                THEN {squared_expr} ELSE NULL END) OVER (
                                PARTITION BY player_name_id
                                ORDER BY season ASC, event ASC
                                ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
                            )
                        END AS {avg_name_squared}""")

        for window in window_sizes:
            # Raw running average
            if include_raw:
                avg_name = f"{short_metric}_{window}"
                select_parts.append(f"""
                    CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                        AVG(CASE WHEN total_points IS NOT NULL
                            THEN {metric} ELSE NULL END) OVER (
                            PARTITION BY player_name_id
                            ORDER BY season ASC, event ASC
                            ROWS BETWEEN {window} PRECEDING AND 1 PRECEDING
                        )
                    END AS {avg_name}""")
            
            # Per-90 running average
            if include_per_90:
                metric_expr = f"CASE WHEN minutes > 0 THEN ({metric} / minutes * 90.0) ELSE 0 END"
                avg_name_per90 = f"{short_metric}_p90_{window}"
                select_parts.append(f"""
                    CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                        AVG(CASE WHEN total_points IS NOT NULL
                            THEN {metric_expr} ELSE NULL END) OVER (
                            PARTITION BY player_name_id
                            ORDER BY season ASC, event ASC
                            ROWS BETWEEN {window} PRECEDING AND 1 PRECEDING
                        )
                    END AS {avg_name_per90}""")
                
                # Squared per-90 running average
                if include_squared_per_90:
                    squared_expr = f"CASE WHEN minutes > 0 THEN POWER(({metric} / minutes * 90.0), 2) ELSE NULL END"
                    avg_name_squared = f"{short_metric}_p90sq_{window}"
                    select_parts.append(f"""
                        CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                            AVG(CASE WHEN total_points IS NOT NULL
                                THEN {squared_expr} ELSE NULL END) OVER (
                                PARTITION BY player_name_id
                                ORDER BY season ASC, event ASC
                                ROWS BETWEEN {window} PRECEDING AND 1 PRECEDING
                            )
                        END AS {avg_name_squared}""")
            
            # ELO-adjusted running average
            if include_elo_adjusted:
                elo_difficulty_multiplier = f"POWER(1.3, (opp_team_elo - team_elo) / 100.0)"
                elo_adjusted_expr = f"{metric} / {elo_difficulty_multiplier}"
                avg_name_elo = f"{short_metric}_elo_{window}"
                select_parts.append(f"""
                    CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                        AVG(CASE WHEN total_points IS NOT NULL
                            THEN {elo_adjusted_expr} ELSE NULL END) OVER (
                            PARTITION BY player_name_id
                            ORDER BY season ASC, event ASC
                            ROWS BETWEEN {window} PRECEDING AND 1 PRECEDING
                        )
                    END AS {avg_name_elo}""")
                
            # ELO-adjusted per-90
            if include_per_90 and include_elo_adjusted:
                elo_per90_expr = f"CASE WHEN minutes > 0 THEN (({metric} / minutes * 90.0) / {elo_difficulty_multiplier}) ELSE 0 END"
                avg_name_elo_per90 = f"{short_metric}_elo_p90_{window}"
                select_parts.append(f"""
                    CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                        AVG(CASE WHEN total_points IS NOT NULL
                            THEN {elo_per90_expr} ELSE NULL END) OVER (
                            PARTITION BY player_name_id
                            ORDER BY season ASC, event ASC
                            ROWS BETWEEN {window} PRECEDING AND 1 PRECEDING
                        )
                    END AS {avg_name_elo_per90}""")
                
                # Squared ELO-adjusted per-90
                if include_squared_per_90:
                    squared_elo_expr = f"CASE WHEN minutes > 0 THEN POWER((({metric} / minutes * 90.0) / {elo_difficulty_multiplier}), 2) ELSE NULL END"
                    avg_name_elo_squared = f"{short_metric}_elo_p90sq_{window}"
                    select_parts.append(f"""
                        CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                            AVG(CASE WHEN total_points IS NOT NULL
                                THEN {squared_elo_expr} ELSE NULL END) OVER (
                                PARTITION BY player_name_id
                                ORDER BY season ASC, event ASC
                                ROWS BETWEEN {window} PRECEDING AND 1 PRECEDING
                            )
                        END AS {avg_name_elo_squared}""")
    
    select_clause = ',\n                        '.join(select_parts)
    
    # Include CTE for latest completed GW
    query = f"""

        WITH latest_gw AS (
            SELECT 
                MAX(event) AS latest_completed_gw
            FROM playergw
            WHERE season = 20252026 AND total_points IS NOT NULL
        ),
        base AS (
            SELECT 
                playergw.*, 
                latest_gw.latest_completed_gw
            FROM playergw
            CROSS JOIN latest_gw
        )

        SELECT *
        FROM (
            SELECT
                {select_clause}
            FROM base
        ) subquery
        WHERE player_season_minutes_total > 0
        ORDER BY player_name_id ASC, season ASC, event ASC
        """
    
    return db_creator.run_sql(query)


# Example usage
metrics = ['minutes',
 'total_points',
 'team_elo',
 'opp_team_elo',
 'goals_scored',
 'bonus',
 'bps',
 'clean_sheets',
 'goals_conceded',
 'expected_goals',
 'expected_assists',
 'expected_goal_involvements',
 'expected_goals_conceded',
 'cbi',
 'defensive_contribution',
 'recoveries',
 'tackles',
 'saves']

# Define interaction pairs that make sense for FPL
interaction_pairs = [
    ('expected_goal_involvements', 'defensive_contribution'),  # All-round 
]

df = create_running_averages(db_creator, 
                             metrics_to_average=metrics,
                             window_sizes=[3,5,10],
                             include_alltime=True,
                             include_per_90=True,
                             include_raw=True,
                             include_elo_adjusted=True,
                             include_squared_per_90=True,
                             interaction_pairs=interaction_pairs)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 147904 entries, 0 to 147903
Columns: 430 entries, player_name_id to saves_elo_p90sq_10
dtypes: float64(425), object(5)
memory usage: 485.2+ MB


In [6]:
import pandas as pd
import numpy as np

current_season = 20252026
current_gw = stats_current_gameweek

# Identify columns to exclude from forward filling (all fpl playergwdf columns) real life data columns
exclude_cols = ['player_name_id', 'season', 'event',
                'fixture', 'total_points', 'minutes',
                'goals_scored', 'assists', 'team_elo', 'opp_team_elo',
                'position', 'bonus', 'bps', 'clean_sheets',
                'goals_conceded', 'was_home', 'expected_assists',
                'expected_goal_involvements', 'expected_goals', 'expected_goals_conceded',
                'starts', 'team_name', 'opp_team_name',
                'cbi','defensive_contribution','recoveries','tackles'
                'saves']

# Create a mask for rows to fill: season 20252026 and event > 10
mask = (df['season'] == current_season) & (df['event'] > current_gw)

# Get columns to fill
cols_to_fill = [col for col in df.columns if col not in exclude_cols]

# Sort by player and event to ensure proper order
df = df.sort_values(['season', 'player_name_id', 'event'])

# Forward fill only for the masked rows
df_filled = df.copy()

for col in cols_to_fill:
    # Forward fill within each player group
    df_filled.loc[mask, col] = df_filled.groupby('player_name_id')[col].ffill()[mask]

In [7]:
mask = (((df_filled['season'] == 20242025) & (df_filled['event'] > 29)) 
    | ((df_filled['season'] == 20252026) & (df_filled['event'] > 0) & (df_filled['event'] < 20))
    ) & (df_filled['player_name_id'] == 'Bukayo Saka')

df_filled.loc[mask, ['player_name_id', 'season', 'event', 'minutes',
                     'total_points', 'opp_team_name', 'opp_team_elo',
                     'expected_goals']].head(50)


,player_name_id,season,event,minutes,total_points,opp_team_name,opp_team_elo,expected_goals
21172,Bukayo Saka,20242025.0,30.0,24.0,8.0,Fulham,1053.766128,0.77
21173,Bukayo Saka,20242025.0,31.0,45.0,1.0,Everton,1020.544178,0.14
21174,Bukayo Saka,20242025.0,32.0,27.0,1.0,Brentford,1024.220969,0.12
21175,Bukayo Saka,20242025.0,33.0,56.0,1.0,Ipswich,929.141346,0.52
21176,Bukayo Saka,20242025.0,35.0,85.0,2.0,Bournemouth,1035.200437,0.48
21177,Bukayo Saka,20242025.0,36.0,87.0,2.0,Liverpool,1254.759550,0.38
21178,Bukayo Saka,20242025.0,37.0,75.0,3.0,Newcastle,1138.718429,0.06
21179,Bukayo Saka,20242025.0,38.0,27.0,1.0,Southampton,811.055997,0.07
21180,Bukayo Saka,20252026.0,1.0,90.0,3.0,Man Utd,1020.755939,0.15
21181,Bukayo Saka,20252026.0,2.0,52.0,6.0,Leeds,935.959117,0.11


In [8]:
#Add a new staging table & data into the database
table_name = "playergw_transformed"

if __name__ == "__main__":
    db_creator.create_staging_table_then_insert_data(table_name, data=df_filled)
    playergw_transformed = db_creator.table_to_df(table_name=table_name)
    print(playergw_transformed.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 147904 entries, 0 to 147903
Columns: 430 entries, player_name_id to saves_elo_p90sq_10
dtypes: float64(425), object(5)
memory usage: 485.2+ MB
None


In [9]:
playergw_transformed.columns.tolist()

['player_name_id',
 'element',
 'season',
 'value',
 'event',
 'minutes',
 'total_points',
 'team_elo',
 'opp_team_elo',
 'position',
 'goals_scored',
 'bonus',
 'bps',
 'clean_sheets',
 'goals_conceded',
 'was_home',
 'expected_goals',
 'expected_assists',
 'expected_goal_involvements',
 'expected_goals_conceded',
 'team_name',
 'opp_team_name',
 'cbi',
 'defensive_contribution',
 'recoveries',
 'tackles',
 'saves',
 'player_season_minutes_total',
 'player_season_points_per90',
 'points_per90',
 'expected_goals_per90',
 'expected_assists_per90',
 'expected_goals_conceded_per90',
 'saves_per90',
 'bps_per90',
 'bonus_per90',
 'cbi_per90',
 'defensive_contribution_per90',
 'recoveries_per90',
 'tackles_per90',
 'elo_diff',
 'xgoal_involvements_x_defensive_cont_p90_at',
 'xgoal_involvements_x_defensive_cont_at',
 'xgoal_involvements_x_defensive_cont_p90_3',
 'xgoal_involvements_x_defensive_cont_3',
 'xgoal_involvements_x_defensive_cont_elo_p90_3',
 'xgoal_involvements_x_defensive_cont_p9

In [10]:
filtered = (playergw_transformed['position'] != 'GKP') & (playergw_transformed['minutes'] > 0)
playergw_transformed[filtered].pivot_table(index=['season'], columns=['position'],
                                 values='expected_goal_involvements', aggfunc='mean')

position,DEF,FWD,GK,MID
season,,,,
20222023.0,0.054594,0.178952,0.001072,0.113314
20232024.0,0.091701,0.319243,0.001449,0.213010
20242025.0,0.085405,0.297725,0.001934,0.193307
20252026.0,0.087005,0.267407,0.001589,0.180040


In [11]:
finish=time.perf_counter()
print(f'Finished in {round(((finish-start)/60),2)} minute(s)')
#Finished in 27.16 minute(s)

Finished in 24.78 minute(s)
